In [74]:
"""
====================================================================
Google Earth Engine - Runoff Prediction Framework
สำหรับการพยากรณ์สมดุลน้ำในลุ่มน้ำน่าน
====================================================================
"""

import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json

# Deep Learning
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import shap

# Initialize GEE
try:
    ee.Initialize(project="ee-sakda-451407")
except Exception as e:
    ee.Authenticate() # หากยังไม่ได้ยืนยันตัวตน ให้ทำการยืนยัน
    ee.Initialize(project="ee-sakda-451407")  

# ====================================================================
# 1. CONFIGURATION & STUDY AREA
# ====================================================================

class Config:
    """Configuration for Nan River Basin Study"""
    
    # Study Area (ลุ่มน้ำน่าน - ปรับ coordinates ตามพื้นที่จริง)
    BASIN_BOUNDARY = ee.Geometry.Polygon([
        [[100.0, 17.5], [101.5, 17.5], 
         [101.5, 19.5], [100.0, 19.5], [100.0, 17.5]]
    ])
    
    # Time Period
    START_DATE = '2014-01-01'
    END_DATE = '2024-12-31'
    
    # Datasets
    CHIRPS = 'UCSB-CHG/CHIRPS/DAILY'
    GPM = 'NASA/GPM_L3/IMERG_V06'
    MOD16 = 'MODIS/006/MOD16A2'
    SSEBOP = 'projects/usgs-ssebop/system:et_fraction'
    MOD11 = 'MODIS/006/MOD11A2'
    MODIS_NDVI = 'MODIS/006/MOD13A2'
    SMAP = 'NASA_USDA/HSL/SMAP10KM_soil_moisture'
    LANDCOVER = 'MODIS/006/MCD12Q1'
    
    # Spatial Resolution
    SCALE = 1000  # meters
    
    # Model Parameters
    LSTM_LOOKBACK = 30  # จำนวนวันย้อนหลังสำหรับ LSTM
    TRAIN_RATIO = 0.8


In [75]:
# show study area on folium map
import folium

def display_study_area():
    """Display the study area on a folium map."""
    basin_coords = Config.BASIN_BOUNDARY.coordinates().getInfo()[0]
    centroid = np.mean(basin_coords, axis=0)[::-1]  # Reverse for folium (lat, lon)

    # add figure height for folium map
    fig = folium.Figure(height=400)

    m = folium.Map(location=centroid, zoom_start=8)
    folium.GeoJson(data=Config.BASIN_BOUNDARY.getInfo(), 
                   style_function=lambda x: {'fillColor': 'blue', 'color': 'blue', 'weight': 2, 'fillOpacity': 0.1}).add_to(m)
    fig.add_child(m)
    return fig

display_study_area()

In [76]:
# ====================================================================
# 2. DATA EXTRACTION FROM GEE
# ====================================================================

class GEEDataExtractor:
    """Extract hydrological data from Google Earth Engine"""
    
    def __init__(self, geometry, start_date, end_date):
        self.geometry = geometry
        self.start_date = start_date
        self.end_date = end_date
        self.scale = Config.SCALE
        
    def extract_precipitation(self, source='CHIRPS'):
        """Extract precipitation data"""
        print("Extracting Precipitation...")
        
        if source == 'CHIRPS':
            dataset = ee.ImageCollection(Config.CHIRPS) \
                .filterDate(self.start_date, self.end_date) \
                .filterBounds(self.geometry)
            band = 'precipitation'
        elif source == 'GPM':
            dataset = ee.ImageCollection(Config.GPM) \
                .filterDate(self.start_date, self.end_date) \
                .filterBounds(self.geometry) \
                .select('precipitationCal')
            band = 'precipitationCal'
        
        return self._reduce_region_timeseries(dataset, band, 'precipitation')
    
    def extract_evapotranspiration(self):
        """Extract ET data (MOD16)"""
        print("Extracting Evapotranspiration...")
        
        dataset = ee.ImageCollection(Config.MOD16) \
            .filterDate(self.start_date, self.end_date) \
            .filterBounds(self.geometry) \
            .select('ET')
        
        # Convert to mm/day and preserve date
        dataset = dataset.map(lambda img: img.multiply(0.1).copyProperties(img, ['system:time_start']))
        
        return self._reduce_region_timeseries(dataset, 'ET', 'et')
    
    def extract_ndvi(self):
        """Extract NDVI (MODIS)"""
        print("Extracting NDVI...")
        
        dataset = ee.ImageCollection(Config.MODIS_NDVI) \
            .filterDate(self.start_date, self.end_date) \
            .filterBounds(self.geometry) \
            .select('NDVI')
        
        # Scale NDVI and preserve date
        dataset = dataset.map(lambda img: img.multiply(0.0001).copyProperties(img, ['system:time_start']))
        
        return self._reduce_region_timeseries(dataset, 'NDVI', 'ndvi')
    
    def extract_lst(self):
        """Extract Land Surface Temperature"""
        print("Extracting LST...")
        
        dataset = ee.ImageCollection(Config.MOD11) \
            .filterDate(self.start_date, self.end_date) \
            .filterBounds(self.geometry) \
            .select('LST_Day_1km')
        
        # Convert to Celsius and preserve date
        dataset = dataset.map(lambda img: img.multiply(0.02).subtract(273.15).copyProperties(img, ['system:time_start']))
        
        return self._reduce_region_timeseries(dataset, 'LST_Day_1km', 'lst')
    
    def extract_soil_moisture(self):
        """Extract Soil Moisture (SMAP)"""
        print("Extracting Soil Moisture...")
        
        dataset = ee.ImageCollection(Config.SMAP) \
            .filterDate(self.start_date, self.end_date) \
            .filterBounds(self.geometry) \
            .select('ssm')
        
        return self._reduce_region_timeseries(dataset, 'ssm', 'soil_moisture')
    
    def extract_land_cover(self, year=2020):
        """Extract dominant land cover type"""
        print("Extracting Land Cover...")
        
        lc = ee.ImageCollection(Config.LANDCOVER) \
            .filterDate(f'{year}-01-01', f'{year}-12-31') \
            .first() \
            .select('LC_Type1')
        
        # Get mode (most common) land cover type
        mode_lc = lc.reduceRegion(
            reducer=ee.Reducer.mode(),
            geometry=self.geometry,
            scale=self.scale,
            maxPixels=1e9
        )
        
        return mode_lc.getInfo()
    
    def _reduce_region_timeseries(self, collection, band, output_name):
        """Helper function to reduce time series to basin mean"""
        
        def reduce_image(img):
            stat = img.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=self.geometry,
                scale=self.scale,
                maxPixels=1e9
            )
            return ee.Feature(None, {
                'date': img.date().format('YYYY-MM-dd'),
                output_name: stat.get(band)
            })
        
        features = collection.map(reduce_image)
        
        # Export to dictionary
        data = features.getInfo()
        
        # Convert to pandas DataFrame
        df = pd.DataFrame([f['properties'] for f in data['features']])
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date').sort_index()
        
        return df

In [77]:
# ====================================================================
# 3. DATA PREPROCESSING
# ====================================================================

class DataPreprocessor:
    """Preprocess and prepare data for modeling"""
    
    @staticmethod
    def merge_datasets(data_dict):
        """Merge all datasets into single DataFrame"""
        print("Merging datasets...")
        
        df_merged = pd.DataFrame()
        for name, df in data_dict.items():
            if df_merged.empty:
                df_merged = df
            else:
                df_merged = df_merged.join(df, how='outer')
        
        return df_merged
    
    @staticmethod
    def fill_missing_values(df, method='interpolate'):
        """Fill missing values"""
        print("Filling missing values...")
        
        if method == 'interpolate':
            df = df.interpolate(method='time', limit_direction='both')
        elif method == 'forward':
            df = df.fillna(method='ffill').fillna(method='bfill')
        
        return df
    
    @staticmethod
    def add_temporal_features(df):
        """Add temporal features"""
        print("Adding temporal features...")
        
        df['month'] = df.index.month
        df['day_of_year'] = df.index.dayofyear
        df['season'] = df['month'].apply(
            lambda x: 1 if x in [3,4,5] else  # Spring
                     2 if x in [6,7,8] else  # Summer/Rainy
                     3 if x in [9,10,11] else  # Autumn
                     4  # Winter
        )
        
        return df
    
    @staticmethod
    def calculate_water_balance_features(df):
        """Calculate water balance related features"""
        print("Calculating water balance features...")
        
        if 'precipitation' in df.columns and 'et' in df.columns:
            df['precip_et_diff'] = df['precipitation'] - df['et']
            df['precip_7d_sum'] = df['precipitation'].rolling(7).sum()
            df['precip_30d_sum'] = df['precipitation'].rolling(30).sum()
            df['et_7d_mean'] = df['et'].rolling(7).mean()
        
        if 'soil_moisture' in df.columns:
            df['sm_change'] = df['soil_moisture'].diff()
        
        return df
    
    @staticmethod
    def add_lag_features(df, target_col='runoff', lags=[1, 2, 3, 7, 14]):
        """Add lag features for target variable"""
        print(f"Adding lag features for {target_col}...")
        
        if target_col in df.columns:
            for lag in lags:
                df[f'{target_col}_lag_{lag}'] = df[target_col].shift(lag)
        
        return df
    
    @staticmethod
    def remove_outliers(df, columns=None, n_std=3):
        """Remove outliers using z-score method"""
        print("Removing outliers...")
        
        if columns is None:
            columns = df.select_dtypes(include=[np.number]).columns
        
        df_clean = df.copy()
        for col in columns:
            if col in df.columns:
                mean = df[col].mean()
                std = df[col].std()
                df_clean = df_clean[(df[col] - mean).abs() <= n_std * std]
        
        print(f"Removed {len(df) - len(df_clean)} outlier rows")
        return df_clean
    
    @staticmethod
    def normalize_data(df, exclude_cols=['runoff'], scale_target=True):
        """Normalize features and optionally target using MinMaxScaler"""
        print("Normalizing data...")
        
        scaler_X = MinMaxScaler()
        scaler_y = MinMaxScaler() if scale_target else None
        
        feature_cols = [col for col in df.columns if col not in exclude_cols]
        
        df_scaled = df.copy()
        df_scaled[feature_cols] = scaler_X.fit_transform(df[feature_cols])
        
        # Scale target variable if specified
        if scale_target and exclude_cols[0] in df.columns:
            df_scaled[exclude_cols[0]] = scaler_y.fit_transform(df[[exclude_cols[0]]])
        
        return df_scaled, scaler_X, scaler_y

In [78]:

# ====================================================================
# 4. FEATURE IMPORTANCE ANALYSIS
# ====================================================================

class FeatureImportance:
    """Analyze feature importance for runoff prediction"""
    
    @staticmethod
    def correlation_analysis(df, target='runoff'):
        """Correlation analysis"""
        print("Performing correlation analysis...")
        
        correlations = df.corr()[target].sort_values(ascending=False)
        
        # Visualization
        plt.figure(figsize=(10, 8))
        sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0)
        plt.title('Correlation Matrix')
        plt.tight_layout()
        plt.show()
        # plt.savefig('correlation_matrix.png', dpi=300)
        # plt.close()
        
        return correlations
    
    @staticmethod
    def random_forest_importance(X, y):
        """Random Forest feature importance"""
        print("Calculating Random Forest importance...")
        
        rf = RandomForestRegressor(n_estimators=100, random_state=42)
        rf.fit(X, y)
        
        importance_df = pd.DataFrame({
            'feature': X.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)
        
        # Visualization
        plt.figure(figsize=(10, 6))
        plt.barh(importance_df['feature'][:10], importance_df['importance'][:10])
        plt.xlabel('Importance')
        plt.title('Top 10 Feature Importance (Random Forest)')
        plt.tight_layout()
        plt.show()
        # plt.savefig('rf_importance.png', dpi=300)
        # plt.close()
        
        return importance_df, rf
    
    @staticmethod
    def shap_analysis(model, X, max_display=10):
        """SHAP values analysis"""
        print("Calculating SHAP values...")
        
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X)
        
        # Summary plot
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X, max_display=max_display, show=False)
        plt.tight_layout()
        plt.show()
        # plt.savefig('shap_summary.png', dpi=300)
        # plt.close()
        
        return shap_values


In [79]:
# ====================================================================
# 5. SIMPLE NEURAL NETWORK FOR RUNOFF PREDICTION
# ====================================================================

class SimpleRunoffModel:
    """Simple feedforward neural network for runoff prediction"""
    
    def __init__(self):
        self.model = None
        self.history = None
        
    def build_model(self, input_dim):
        """Build simple feedforward neural network"""
        print(f"Building simple NN model with {input_dim} input features...")
        
        model = Sequential([
            Dense(32, activation='relu', input_dim=input_dim),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dropout(0.2),
            Dense(8, activation='relu'),
            Dense(1, activation='linear')
        ])
        
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=0.001), 
            loss='mse', 
            metrics=['mae']
        )
        self.model = model
        
        print(model.summary())
        return model
    
    def train(self, X_train, y_train, X_val, y_val, epochs=200, batch_size=32):
        """Train the model"""
        print("Training model...")
        
        print(f"Training data shape: X={X_train.shape}, y={y_train.shape}")
        print(f"Validation data shape: X={X_val.shape}, y={y_val.shape}")
        
        # Build model if not exists
        if self.model is None:
            self.build_model(X_train.shape[1])
        
        # Callbacks
        early_stop = keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=40, restore_best_weights=True, verbose=1
        )
        
        reduce_lr = keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=20, min_lr=0.00001, verbose=1
        )
        
        # Train
        self.history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[early_stop, reduce_lr],
            verbose=1
        )
        
        return self.history
    
    def predict(self, X):
        """Make predictions"""
        return self.model.predict(X, verbose=0)
    
    def evaluate(self, X_test, y_test, scaler_y=None):
        """Evaluate model performance"""
        print("Evaluating model...")
        
        y_pred = self.predict(X_test)
        y_true = y_test
        
        # Inverse transform if scaler provided
        if scaler_y is not None:
            y_true_orig = scaler_y.inverse_transform(y_true.reshape(-1, 1)).flatten()
            y_pred_orig = scaler_y.inverse_transform(y_pred).flatten()
        else:
            y_true_orig = y_true.flatten()
            y_pred_orig = y_pred.flatten()
        
        # Metrics on original scale
        rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
        mae = np.mean(np.abs(y_true_orig - y_pred_orig))
        r2 = r2_score(y_true_orig, y_pred_orig)
        
        # NSE
        numerator = np.sum((y_true_orig - y_pred_orig)**2)
        denominator = np.sum((y_true_orig - np.mean(y_true_orig))**2)
        nse = 1 - (numerator / denominator) if denominator != 0 else -999
        
        # MAPE
        mape = np.mean(np.abs((y_true_orig - y_pred_orig) / (y_true_orig + 1e-10))) * 100
        
        # Correlation coefficient
        corr = np.corrcoef(y_true_orig, y_pred_orig)[0, 1]
        
        metrics = {
            'RMSE': rmse,
            'MAE': mae,
            'R²': r2,
            'NSE': nse,
            'MAPE': mape,
            'Correlation': corr
        }
        
        print(f"\n{'='*60}")
        print(f"Model Performance Metrics:")
        print(f"{'='*60}")
        print(f"RMSE:        {rmse:.4f}")
        print(f"MAE:         {mae:.4f}")
        print(f"R²:          {r2:.4f}")
        print(f"NSE:         {nse:.4f}")
        print(f"MAPE:        {mape:.2f}%")
        print(f"Correlation: {corr:.4f}")
        print(f"{'='*60}\n")
        
        # Performance interpretation
        if nse > 0.75:
            perf = "Very Good ✓"
        elif nse > 0.65:
            perf = "Good"
        elif nse > 0.50:
            perf = "Satisfactory"
        else:
            perf = "Unsatisfactory - Consider using actual runoff data"
        
        print(f"Performance Rating: {perf}\n")
        
        return metrics, y_pred_orig, y_true_orig
    
    def plot_results(self, y_true, y_pred, dates=None):
        """Plot prediction results"""
        
        fig, axes = plt.subplots(2, 1, figsize=(14, 10))
        
        # Time series plot
        if dates is not None:
            axes[0].plot(dates, y_true, label='Observed', alpha=0.8, linewidth=1.5, color='blue')
            axes[0].plot(dates, y_pred, label='Predicted', alpha=0.8, linewidth=1.5, color='red')
        else:
            axes[0].plot(y_true, label='Observed', alpha=0.8, color='blue')
            axes[0].plot(y_pred, label='Predicted', alpha=0.8, color='red')
        
        axes[0].set_xlabel('Time', fontsize=12)
        axes[0].set_ylabel('Runoff', fontsize=12)
        axes[0].set_title('Runoff Prediction - Time Series', fontsize=14, fontweight='bold')
        axes[0].legend(fontsize=11)
        axes[0].grid(True, alpha=0.3)
        
        # Scatter plot
        axes[1].scatter(y_true, y_pred, alpha=0.5, s=30, color='steelblue')
        
        # Add 1:1 line
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        axes[1].plot([min_val, max_val], [min_val, max_val], 
                     'r--', lw=2, label='1:1 Line')
        
        # Add R² to plot
        r2 = r2_score(y_true, y_pred)
        axes[1].text(0.05, 0.95, f'R² = {r2:.3f}', 
                    transform=axes[1].transAxes, fontsize=12,
                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        axes[1].set_xlabel('Observed Runoff', fontsize=12)
        axes[1].set_ylabel('Predicted Runoff', fontsize=12)
        axes[1].set_title('Observed vs Predicted', fontsize=14, fontweight='bold')
        axes[1].legend(fontsize=11)
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        return fig

In [80]:
# ====================================================================
# 6. MAIN WORKFLOW - SIMPLIFIED (NO LSTM)
# ====================================================================

def main_workflow():
    """Main research workflow - Simplified approach"""
    
    print("="*60)
    print("RUNOFF PREDICTION WORKFLOW - ลุ่มน้ำน่าน")
    print("Using Simple Neural Network (Non-LSTM)")
    print("="*60)
    
    # Initialize extractor
    extractor = GEEDataExtractor(
        Config.BASIN_BOUNDARY,
        Config.START_DATE,
        Config.END_DATE
    )
    
    # Step 1: Extract data from GEE
    print("\n[STEP 1] Extracting data from GEE...")
    data_dict = {
        'precipitation': extractor.extract_precipitation(),
        'et': extractor.extract_evapotranspiration(),
        'ndvi': extractor.extract_ndvi(),
        'lst': extractor.extract_lst(),
        'soil_moisture': extractor.extract_soil_moisture()
    }
    
    # Step 2: Merge datasets
    print("\n[STEP 2] Merging and aligning datasets...")
    preprocessor = DataPreprocessor()
    df = preprocessor.merge_datasets(data_dict)
    df = preprocessor.fill_missing_values(df)
    
    print(f"Merged dataset shape: {df.shape}")
    
    # Step 3: Generate realistic synthetic runoff
    print("\n[STEP 3] Generating synthetic runoff data...")
    
    if 'precipitation' in df.columns and 'et' in df.columns:
        precip = df['precipitation'].values
        et = df['et'].values
        
        # Water balance with strong correlation
        alpha = 0.5
        baseflow = 12
        
        # Add trend and seasonality
        month = pd.DatetimeIndex(df.index).month
        seasonal = 1 + 0.4 * np.sin(2 * np.pi * (month - 3) / 12)
        
        water_balance = (precip - et) * seasonal
        synthetic_runoff = np.maximum(3, alpha * water_balance + baseflow + np.random.normal(0, 2, len(precip)))
        
        df['runoff'] = synthetic_runoff
    else:
        raise ValueError("Precipitation or ET data missing!")
    
    # Step 4: Feature engineering
    print("\n[STEP 4] Feature engineering...")
    df = preprocessor.add_temporal_features(df)
    df = preprocessor.calculate_water_balance_features(df)
    
    # Add lag features for runoff
    df = preprocessor.add_lag_features(df, target_col='runoff', lags=[1, 2, 3, 7])
    
    # Remove NaN
    df = df.dropna()
    
    # Light outlier removal
    df = preprocessor.remove_outliers(df, columns=['runoff'], n_std=4)
    
    print(f"Dataset shape after preprocessing: {df.shape}")
    print(f"\nRunoff statistics:")
    print(df['runoff'].describe())
    
    # Step 5: Feature Selection
    print("\n[STEP 5] Feature selection...")
    
    X_all = df.drop('runoff', axis=1)
    y = df['runoff']
    
    # Use Random Forest for feature importance
    rf_temp = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_temp.fit(X_all, y)
    
    feature_importance = pd.DataFrame({
        'feature': X_all.columns,
        'importance': rf_temp.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Select top features
    top_features = feature_importance.head(12)['feature'].tolist()
    print(f"\nTop 12 features selected:")
    for i, feat in enumerate(top_features, 1):
        imp = feature_importance[feature_importance['feature'] == feat]['importance'].values[0]
        print(f"  {i}. {feat}: {imp:.4f}")
    
    # Create reduced dataset
    df_reduced = df[top_features + ['runoff']].copy()
    print(f"\nReduced dataset shape: {df_reduced.shape}")
    
    # Step 6: Prepare data
    print("\n[STEP 6] Preparing data for training...")
    
    # Normalize
    df_scaled, scaler_X, scaler_y = preprocessor.normalize_data(
        df_reduced, exclude_cols=['runoff'], scale_target=True
    )
    
    # Temporal split
    train_size = int(len(df_scaled) * Config.TRAIN_RATIO)
    train_data = df_scaled[:train_size]
    test_data = df_scaled[train_size:]
    
    X_train = train_data.drop('runoff', axis=1).values
    y_train = train_data['runoff'].values
    X_test = test_data.drop('runoff', axis=1).values
    y_test = test_data['runoff'].values
    
    print(f"Train set: {len(X_train)} samples")
    print(f"Test set: {len(X_test)} samples")
    
    # Step 7: Train Model
    print("\n[STEP 7] Training Simple Neural Network...")
    model = SimpleRunoffModel()
    
    # Split train into train and validation
    val_size = int(len(X_train) * 0.2)
    X_val = X_train[-val_size:]
    y_val = y_train[-val_size:]
    X_train_final = X_train[:-val_size]
    y_train_final = y_train[:-val_size]
    
    # Train
    history = model.train(X_train_final, y_train_final, X_val, y_val, 
                         epochs=200, batch_size=32)
    
    # Step 8: Evaluate
    print("\n[STEP 8] Evaluating model...")
    metrics, y_pred, y_true = model.evaluate(X_test, y_test, scaler_y=scaler_y)
    
    # Plot results
    test_dates = df_scaled.index[train_size:]
    model.plot_results(y_true, y_pred, dates=test_dates)
    
    # Step 9: Plot training history
    print("\n[STEP 9] Plotting training history...")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss (MSE)', fontsize=12)
    axes[0].set_title('Training History - Loss', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)
    axes[1].plot(history.history['val_mae'], label='Val MAE', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('MAE', fontsize=12)
    axes[1].set_title('Training History - MAE', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Step 10: Save results
    print("\n[STEP 10] Saving results...")
    
    results = {
        'metrics': metrics,
        'top_features': top_features,
        'data_shape': str(df_reduced.shape),
        'train_size': train_size,
        'test_size': len(df_scaled) - train_size
    }
    
    with open('results_summary.json', 'w') as f:
        json.dump({k: str(v) for k, v in results.items()}, f, indent=2)
    
    print("\n" + "="*60)
    print("WORKFLOW COMPLETED!")
    print("="*60)
    
    return df_reduced, model, metrics, scaler_y, top_features

In [ ]:
# ====================================================================
# EXAMPLE USAGE
# ====================================================================

if __name__ == "__main__":
    # Run complete workflow
    df, model, metrics, scaler_y, top_features = main_workflow()
    
    print("\n✓ Research framework completed successfully!")
    print(f"\nFinal Model Performance:")
    print(f"  NSE = {metrics['NSE']:.4f}")
    print(f"  R²  = {metrics['R²']:.4f}")
    print(f"\nTop features used: {top_features}")